# Exploratory Data Analysis (EDA)
This notebook performs basic EDA on the first available CSV found in the repository `data/processed`. It reports shape, types, missing values, basic summary statistics, and simple plots for numeric columns.

**Author: Mehreen Ali Gillani**

In [6]:
# Imports
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# Set up paths

# Define repo_root so notebook can locate data relative to repository root
repo_root = Path().resolve()
while not (repo_root / '.git').exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent

data_path = repo_root / 'data' / 'processed' / 'nyc_annual_salary_employees_payBasis_perAnuum.csv'

df_original = pd.read_csv(data_path, low_memory=False)

# Display first rows
df_original.head()

,Fiscal Year,Payroll Number,Agency Name,Last Name,First Name,Agency Start Date,Work Location Borough,Title Description,Leave Status as of June 30,Base Salary,Pay Basis,Regular Hours,Regular Gross Paid,OT Hours,Total OT Paid,Total Other Pay,Employee_Agency_Tenure,Agency_Std,Title_Category,Title_Std
0,2020,17.0,OFFICE OF EMERGENCY MANAGEMENT,BEREZIN,MIKHAIL,2015-08-10,BROOKLYN,EMERGENCY PREPAREDNESS MANAGER,ACTIVE,86005.0,per Annum,1820.0,84698.21,0.0,0.0,0.0,4.889802,Office Of Emergency Management,Management,Emergency Preparedness Manager
1,2020,17.0,OFFICE OF EMERGENCY MANAGEMENT,GEAGER,VERONICA,2016-09-12,BROOKLYN,EMERGENCY PREPAREDNESS MANAGER,ACTIVE,86005.0,per Annum,1820.0,84698.21,0.0,0.0,0.0,3.797399,Office Of Emergency Management,Management,Emergency Preparedness Manager
2,2020,17.0,OFFICE OF EMERGENCY MANAGEMENT,RAMANI,SHRADDHA,2016-02-22,BROOKLYN,EMERGENCY PREPAREDNESS MANAGER,ACTIVE,86005.0,per Annum,1820.0,84698.21,0.0,0.0,0.0,4.353183,Office Of Emergency Management,Management,Emergency Preparedness Manager
3,2020,17.0,OFFICE OF EMERGENCY MANAGEMENT,ROTTA,JONATHAN,2013-09-16,BROOKLYN,EMERGENCY PREPAREDNESS MANAGER,ACTIVE,86005.0,per Annum,1820.0,84698.21,0.0,0.0,0.0,6.787132,Office Of Emergency Management,Management,Emergency Preparedness Manager
4,2020,17.0,OFFICE OF EMERGENCY MANAGEMENT,WILSON II,ROBERT,2018-04-30,BROOKLYN,EMERGENCY PREPAREDNESS MANAGER,ACTIVE,86005.0,per Annum,1820.0,84698.21,0.0,0.0,0.0,2.168378,Office Of Emergency Management,Management,Emergency Preparedness Manager


In [7]:
# Delete the 'Pay Basis' column from df_original
df_original = df_original.drop(columns=['Pay Basis', 'Leave Status as of June 30'])

In [8]:
df_original.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2770022 entries, 0 to 2770021
Data columns (total 18 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   Fiscal Year             int64  
 1   Payroll Number          float64
 2   Agency Name             object 
 3   Last Name               object 
 4   First Name              object 
 5   Agency Start Date       object 
 6   Work Location Borough   object 
 7   Title Description       object 
 8   Base Salary             float64
 9   Regular Hours           float64
 10  Regular Gross Paid      float64
 11  OT Hours                float64
 12  Total OT Paid           float64
 13  Total Other Pay         float64
 14  Employee_Agency_Tenure  float64
 15  Agency_Std              object 
 16  Title_Category          object 
 17  Title_Std               object 
dtypes: float64(8), int64(1), object(9)
memory usage: 380.4+ MB


In [9]:
# Calculate Total_compensation
df_original['Total_Compensation'] = (
    df_original['Regular Gross Paid'].fillna(0) + 
    df_original['Total OT Paid'].fillna(0) + 
    df_original['Total Other Pay'].fillna(0)
)

In [10]:
# Flag where Total Compensation is surprisingly lower than Base Salary
anomalies = df_original[df_original['Total_Compensation'] < df_original['Base Salary'] * 0.5]
print(f"Employees where Total Comp < 50% of Base Salary: {len(anomalies):,}")

Employees where Total Comp < 50% of Base Salary: 79,110


In [11]:
anomalies = df_original[df_original['Total_Compensation'] < df_original['Base Salary'] * 0.5].copy()

# Quick profile
print(f"Anomaly count: {len(anomalies):,}")
print(f"% of total workforce: {len(anomalies)/len(df_original)*100:.1f}%")
print(f"\n--- Compensation Stats for Anomalies ---")
print(anomalies[['Base Salary', 'Regular Gross Paid', 'Total OT Paid', 
                  'Total Other Pay', 'Total_Compensation']].describe().round(2))

Anomaly count: 79,110
% of total workforce: 2.9%

--- Compensation Stats for Anomalies ---
       Base Salary  Regular Gross Paid  Total OT Paid  Total Other Pay  \
count     79110.00            79110.00       79110.00         79110.00   
mean      55995.98            15069.16         194.10           391.74   
std       26440.62            11440.07         686.68          1166.51   
min         369.53                0.00           0.00             0.00   
25%       40369.00             7080.20           0.00             0.00   
50%       47394.00            13311.48           0.00             0.00   
75%       63751.00            20027.43           0.00           180.53   
max      345000.00           129996.75       18680.36         38749.96   

       Total_Compensation  
count            79110.00  
mean             15655.00  
std              11717.95  
min                  0.96  
25%               7402.24  
50%              13980.47  
75%              20477.95  
max             12

In [12]:
# Confirm with hours analysis
print(anomalies['Regular Hours'].describe())
print("\nHours distribution:")
bins = [0, 100, 500, 1000, 2080]
labels = ['Very Low (<100h)', 'Low (100-500h)', 'Partial (500-1000h)', 'Full Year (1000+h)']
anomalies['Hours_Bucket'] = pd.cut(anomalies['Regular Hours'], bins=bins, labels=labels)
print(anomalies['Hours_Bucket'].value_counts())

count    79110.000000
mean       473.367482
std        339.690200
min          0.000000
25%        175.000000
50%        455.000000
75%        740.000000
max       2828.000000
Name: Regular Hours, dtype: float64

Hours distribution:
Hours_Bucket
Partial (500-1000h)    34396
Low (100-500h)         28899
Very Low (<100h)        5464
Full Year (1000+h)      1555
Name: count, dtype: int64


In [13]:
df_full = df_original.copy()

In [14]:
# print("COMPREHENSIVE DATA CLEANING PIPELINE")
# print("=" * 60)

# 1. Start with original
print(f"1. Original dataset: {len(df_full):,} records")

# 2. Remove zero/negative compensation
initial_count = len(df_full)
df_full = df_full[df_full['Total_Compensation'] > 0].copy()
print(f"2. Removed zero/negative compensation: {initial_count - len(df_full):,} records")

# 3. Flag employee types based on hours tracking
df_full['Hours_Tracked'] = df_full['Regular Hours'] > 0

print(f"\n3. MEANINGFUL EMPLOYMENT FILTER:")

# A. Hourly employees: worked meaningful hours + reasonable pay
# 1000h threshold confirmed by our anomaly analysis
hourly_candidates = df_full[
    (df_full['Hours_Tracked']) &
    (df_full['Regular Hours'] >= 1000) &       # ~6 months minimum
    (df_full['Total_Compensation'] >= 20000)    # ~$10/hr minimum
].copy()

# B. Salaried employees: no hours tracked but reasonable pay
# Raised threshold slightly - $30k is reasonable minimum annual salary
salaried_candidates = df_full[
    (~df_full['Hours_Tracked']) &
    (df_full['Total_Compensation'] >= 30000)
].copy()

# NOTE: Removed special_cases - already captured in salaried_candidates
# Police/Fire/Correction with $50k+ are a subset of salaried ($30k+)

# Combine
df = pd.concat([hourly_candidates, salaried_candidates]).drop_duplicates()

print(f"   Hourly   (hours >= 1000h, comp >= $20k): {len(hourly_candidates):,}")
print(f"   Salaried (no hours,       comp >= $30k): {len(salaried_candidates):,}")
print(f"   Total meaningful: {len(df):,} records")
print(f"   Removed: {len(df_full) - len(df):,} ({(len(df_full)-len(df))/len(df_full)*100:.1f}%)")

# 4. Part-year flag for optional filtering later
df['Employment_Type'] = pd.cut(
    df['Regular Hours'],
    bins=[0, 100, 500, 1000, float('inf')],
    labels=['Very_Low_Hours', 'Part_Year', 'Partial_Year', 'Full_Year'],
    include_lowest=True
)

# 5. Final datasets
print(f"\n4. FINAL ANALYSIS DATASETS:")
df_hourly_final  = df[df['Hours_Tracked']].copy()
df_salaried_final = df[~df['Hours_Tracked']].copy()
print(f"   df_hourly_final:   {len(df_hourly_final):,} records")
print(f"   df_salaried_final: {len(df_salaried_final):,} records")
print(f"   df (combined):     {len(df):,} records")

# 6. Data quality check
print(f"\n5. FINAL DATA QUALITY CHECK:")
print(f"   Avg compensation: ${df['Total_Compensation'].mean():,.0f}")
print(f"   Median comp:      ${df['Total_Compensation'].median():,.0f}")
print(f"   Min compensation: ${df['Total_Compensation'].min():,.0f}")
print(f"   Max compensation: ${df['Total_Compensation'].max():,.0f}")
print(f"   Years: {df['Fiscal Year'].min()} to {df['Fiscal Year'].max()}")
print(f"   Unique agencies:  {df['Agency_Std'].nunique()}")
print(f"   Hourly workers:   {df['Hours_Tracked'].sum():,}")
print(f"   Salaried workers: {(~df['Hours_Tracked']).sum():,}")

print(f"\n{'='*60}")
print("CLEANING COMPLETE - READY FOR ANALYSIS")
print(f"{'='*60}")

# Convenience aliases
df_hourly   = df_hourly_final.copy()
df_salaried = df_salaried_final.copy()

1. Original dataset: 2,770,022 records
2. Removed zero/negative compensation: 0 records

3. MEANINGFUL EMPLOYMENT FILTER:
   Hourly   (hours >= 1000h, comp >= $20k): 1,586,600
   Salaried (no hours,       comp >= $30k): 1,031,051
   Total meaningful: 2,617,642 records
   Removed: 152,380 (5.5%)

4. FINAL ANALYSIS DATASETS:
   df_hourly_final:   1,586,600 records
   df_salaried_final: 1,031,042 records
   df (combined):     2,617,642 records

5. FINAL DATA QUALITY CHECK:
   Avg compensation: $88,846
   Median comp:      $81,999
   Min compensation: $20,000
   Max compensation: $1,689,518
   Years: 2015 to 2024
   Unique agencies:  148
   Hourly workers:   1,586,600
   Salaried workers: 1,031,042

CLEANING COMPLETE - READY FOR ANALYSIS


In [15]:
# Who are the top earners? Verify they're legitimate
top_earners = df.nlargest(20, 'Total_Compensation')[
    ['Agency_Std', 'Title_Std', 'Title_Category', 
     'Base Salary', 'Total OT Paid', 'Total_Compensation', 'Fiscal Year']
]
print(top_earners.to_string())

# Check compensation distribution above $500k
print(f"\nEarners above $500k:   {(df['Total_Compensation'] > 500000).sum():,}")
print(f"Earners above $300k:   {(df['Total_Compensation'] > 300000).sum():,}")
print(f"Earners above $200k:   {(df['Total_Compensation'] > 200000).sum():,}")

                           Agency_Std                                      Title_Std  Title_Category  Base Salary  Total OT Paid  Total_Compensation  Fiscal Year
2748285  Department of Transportation                          Chief Marine Engineer      Management     169520.0      498669.50          1689517.92         2024
2748282  Department of Transportation                          Chief Marine Engineer      Management     169520.0      423152.06          1559298.82         2024
2748283  Department of Transportation                          Chief Marine Engineer      Management     169520.0      431305.66          1527970.79         2024
2748276  Department of Transportation                          Chief Marine Engineer      Management     169520.0      479668.38          1437385.60         2024
2748279  Department of Transportation                          Chief Marine Engineer      Management     169520.0      327558.22          1417636.64         2024
2748273  Department of Trans

In [16]:
def get_borough_distribution(df, dataset_name):
    """Get borough distribution with proper formatting"""
    # Handle missing values
    total = len(df)
    counts = df['Work Location Borough'].value_counts(dropna=False)
    percentages = (counts / total * 100).round(2)
    
    print(f"\n{dataset_name} - Borough Distribution:")
    print("-" * 50)
    for borough, count in counts.items():
        percent = percentages[borough]
        borough_label = str(borough) if borough == borough else 'UNKNOWN/NaN'  # handle NaN
        print(f"  {borough_label:20} {count:>10,} records ({percent:>6.2f}%)")
    print(f"  {'TOTAL':20} {total:>10,} records (100.00%)")
    return counts

print("=" * 60)
print("BOROUGH DISTRIBUTION ACROSS DATASETS")
print("=" * 60)

df_counts       = get_borough_distribution(df, "COMBINED (df)")
salaried_counts = get_borough_distribution(df_salaried_final, "SALARIED EMPLOYEES")
hourly_counts   = get_borough_distribution(df_hourly_final,   "HOURLY EMPLOYEES")

# Borough comparison
print("\n" + "=" * 60)
print("BOROUGH COMPARISON: SALARIED vs HOURLY")
print("=" * 60)
print(f"  {'Borough':20} {'Salaried':>10} {'Hourly':>10} {'Diff':>8}")
print("-" * 55)

for borough in sorted(df['Work Location Borough'].dropna().unique()):
    salaried_pct = round(salaried_counts.get(borough, 0) / len(df_salaried_final) * 100, 2)
    hourly_pct   = round(hourly_counts.get(borough, 0)   / len(df_hourly_final)   * 100, 2)
    diff         = hourly_pct - salaried_pct
    
    # Flag large differences
    flag = " ⚠️" if abs(diff) > 5 else ""
    print(f"  {str(borough):20} {salaried_pct:>9.2f}% {hourly_pct:>9.2f}% {diff:>+7.1f}%{flag}")

# Summary stats
print(f"\n{'=' * 60}")
print("MISSING BOROUGH DATA:")
print(f"  Combined:  {df['Work Location Borough'].isna().sum():,} nulls "
      f"({df['Work Location Borough'].isna().mean()*100:.2f}%)")
print(f"  Salaried:  {df_salaried_final['Work Location Borough'].isna().sum():,} nulls "
      f"({df_salaried_final['Work Location Borough'].isna().mean()*100:.2f}%)")
print(f"  Hourly:    {df_hourly_final['Work Location Borough'].isna().sum():,} nulls "
      f"({df_hourly_final['Work Location Borough'].isna().mean()*100:.2f}%)")

BOROUGH DISTRIBUTION ACROSS DATASETS

COMBINED (df) - Borough Distribution:
--------------------------------------------------
  MANHATTAN             1,658,306 records ( 63.35%)
  QUEENS                  374,468 records ( 14.31%)
  BROOKLYN                345,983 records ( 13.22%)
  BRONX                   186,682 records (  7.13%)
  STATEN ISLAND            52,203 records (  1.99%)
  TOTAL                 2,617,642 records (100.00%)

SALARIED EMPLOYEES - Borough Distribution:
--------------------------------------------------
  MANHATTAN             1,031,041 records (100.00%)
  QUEENS                        1 records (  0.00%)
  TOTAL                 1,031,042 records (100.00%)

HOURLY EMPLOYEES - Borough Distribution:
--------------------------------------------------
  MANHATTAN               627,265 records ( 39.54%)
  QUEENS                  374,467 records ( 23.60%)
  BROOKLYN                345,983 records ( 21.81%)
  BRONX                   186,682 records ( 11.77%)
  STATEN 

In [17]:
# Check what agencies make up the salaried dataset
print("Salaried employees by agency:")
print(df_salaried_final['Agency_Std'].value_counts().head(20))

print("\nSalaried employees by Title Category:")
print(df_salaried_final['Title_Category'].value_counts().head(10))

# Check if Manhattan = default/placeholder value
print("\nHourly Manhattan agencies (for comparison):")
print(df_hourly_final[df_hourly_final['Work Location Borough'] == 'MANHATTAN']['Agency_Std'].value_counts().head(10))

print("\nSalaried Manhattan agencies:")
print(df_salaried_final[df_salaried_final['Work Location Borough'] == 'MANHATTAN']['Agency_Std'].value_counts().head(10))

Salaried employees by agency:
Agency_Std
Department of Education    1031041
Police Department                1
Name: count, dtype: int64

Salaried employees by Title Category:
Title_Category
Education - Teaching    751450
Education - Support     211950
Other Roles              57658
Administrative            8402
Management                 978
Technical                  593
Medical/Health               8
Unknown Role                 2
Police                       1
Name: count, dtype: int64

Hourly Manhattan agencies (for comparison):
Agency_Std
Police Department                         172773
Department of Education                    71540
Hra/Dept Of Social Services                46770
Fire Department                            32119
Nyc Housing Authority                      31192
Administration for Children's Services     29078
Department of Sanitation                   22075
Housing Preservation & Dvlpmnt             17340
Department of Health                       16791
Departm

Salaried = 100% Department of Education (DOE)
DOE = 1,031,041 records all coded as MANHATTAN

In [18]:
# DOE appears in BOTH hourly and salaried - understand the split
doe_hourly   = df_hourly_final[df_hourly_final['Agency_Std'] == 'Department of Education']
doe_salaried = df_salaried_final[df_salaried_final['Agency_Std'] == 'Department of Education']

print(f"DOE Hourly   (hours tracked): {len(doe_hourly):,}")
print(f"DOE Salaried (no hours):      {len(doe_salaried):,}")
print(f"DOE Total:                    {len(doe_hourly) + len(doe_salaried):,}")

print(f"\nDOE Hourly Title Categories:")
print(doe_hourly['Title_Category'].value_counts())

print(f"\nDOE Salaried Title Categories:")
print(doe_salaried['Title_Category'].value_counts())

# Borough analysis - exclude DOE entirely
df_borough_valid = df_hourly_final[
    df_hourly_final['Agency_Std'] != 'Department of Education'
].copy()

print(f"Valid borough records (non-DOE hourly): {len(df_borough_valid):,}")
print(f"Removed DOE hourly records: {len(doe_hourly):,}")

# Now borough distribution is meaningful
print("\nCLEAN Borough Distribution (Non-DOE Hourly):")
borough_clean = get_borough_distribution(df_borough_valid, "NON-DOE HOURLY")

DOE Hourly   (hours tracked): 104,769
DOE Salaried (no hours):      1,031,041
DOE Total:                    1,135,810

DOE Hourly Title Categories:
Title_Category
Other Roles             58773
Administrative          20379
Technical                9893
Management               7513
Medical/Health           5948
Education - Teaching     1946
Education - Support       287
Unknown Role               30
Name: count, dtype: int64

DOE Salaried Title Categories:
Title_Category
Education - Teaching    751450
Education - Support     211950
Other Roles              57658
Administrative            8402
Management                 978
Technical                  593
Medical/Health               8
Unknown Role                 2
Name: count, dtype: int64
Valid borough records (non-DOE hourly): 1,481,831
Removed DOE hourly records: 104,769

CLEAN Borough Distribution (Non-DOE Hourly):

NON-DOE HOURLY - Borough Distribution:
--------------------------------------------------
  MANHATTAN               5

In [19]:
# Segment 1: DOE employees (teaching + support) - no borough
df_doe = df[df['Agency_Std'] == 'Department of Education'].copy()

# Segment 2: Uniformed services - hourly, borough valid
uniformed = ['Police Department', 'Fire Department', 
             'Department of Correction', 'Department of Sanitation']
df_uniformed = df_hourly_final[df_hourly_final['Agency_Std'].isin(uniformed)].copy()

# Segment 3: All other city agencies - hourly, borough valid
df_civilian = df_hourly_final[
    (~df_hourly_final['Agency_Std'].isin(uniformed)) &
    (df_hourly_final['Agency_Std'] != 'Department of Education')
].copy()

print(f"DOE employees:       {len(df_doe):,}")
print(f"Uniformed services:  {len(df_uniformed):,}")
print(f"Civilian agencies:   {len(df_civilian):,}")
print(f"Total:               {len(df_doe)+len(df_uniformed)+len(df_civilian):,}")

DOE employees:       1,135,810
Uniformed services:  802,348
Civilian agencies:   679,483
Total:               2,617,641


In [20]:
# Should be 2,617,642 but got 2,617,641
# Likely the 1 Police Department record in salaried
print("Check: 1 Police salaried record")
police_salaried = df_salaried_final[df_salaried_final['Agency_Std'] == 'Police Department']
print(police_salaried[['Agency_Std', 'Title_Std', 'Total_Compensation', 'Regular Hours']])
# This record has no hours so went to salaried but isn't in df_uniformed (hourly only)

Check: 1 Police salaried record
                Agency_Std       Title_Std  Total_Compensation  Regular Hours
2499766  Police Department  Police Officer             31676.2            0.0


In [21]:
# Simply pull uniformed from df (all records) not df_hourly_final
df_uniformed = df[df['Agency_Std'].isin(uniformed)].copy()

# Civilian = everyone except DOE and uniformed
df_civilian = df[
    (~df['Agency_Std'].isin(uniformed)) &
    (df['Agency_Std'] != 'Department of Education')
].copy()

# Verify totals
total = len(df_doe) + len(df_uniformed) + len(df_civilian)
print(f"DOE employees:       {len(df_doe):,}")
print(f"Uniformed services:  {len(df_uniformed):,}")
print(f"Civilian agencies:   {len(df_civilian):,}")
print(f"Total:               {total:,}")
print(f"Match original:      {total == len(df)} ✅" if total == len(df) else f"Still missing: {len(df) - total}")

DOE employees:       1,135,810
Uniformed services:  802,349
Civilian agencies:   679,483
Total:               2,617,642
Match original:      True ✅


In [22]:
segments = {'DOE': df_doe, 'Uniformed': df_uniformed, 'Civilian': df_civilian}

print(f"\n{'='*70}")
print(f"{'SEGMENT PROFILES':^70}")
print(f"{'='*70}")
print(f"{'Metric':<25} {'DOE':>13} {'Uniformed':>13} {'Civilian':>13}")
print(f"{'-'*70}")

metrics = {
    'Record Count':     lambda d: f"{len(d):>13,}",
    'Avg Compensation': lambda d: f"${d['Total_Compensation'].mean():>12,.0f}",
    'Median Comp':      lambda d: f"${d['Total_Compensation'].median():>12,.0f}",
    'Max Comp':         lambda d: f"${d['Total_Compensation'].max():>12,.0f}",
    'Avg Base Salary':  lambda d: f"${d['Base Salary'].mean():>12,.0f}",
    'Avg OT Paid':      lambda d: f"${d['Total OT Paid'].mean():>12,.0f}",
    'OT % of Comp':     lambda d: f"{(d['Total OT Paid'].sum()/d['Total_Compensation'].sum()*100):>12.1f}%",
    'Unique Agencies':  lambda d: f"{d['Agency_Std'].nunique():>13}",
    'Years Covered':    lambda d: f"  {d['Fiscal Year'].min()}-{d['Fiscal Year'].max()}",
}

for metric, func in metrics.items():
    row = f"{metric:<25}"
    for seg in segments.values():
        row += func(seg)
    print(row)

print(f"{'='*70}")


                           SEGMENT PROFILES                           
Metric                              DOE     Uniformed      Civilian
----------------------------------------------------------------------
Record Count                 1,135,810      802,349      679,483
Avg Compensation         $      83,581$     106,722$      76,540
Median Comp              $      82,324$     106,974$      69,021
Max Comp                 $     448,270$     478,302$   1,689,518
Avg Base Salary          $      82,411$      76,782$      70,922
Avg OT Paid              $         131$      18,350$       4,327
OT % of Comp                      0.2%        17.2%         5.7%
Unique Agencies                      1            4          143
Years Covered              2015-2024  2015-2024  2015-2024


Over time analysis

In [23]:
print("=" * 60)
print("OVERTIME ANALYSIS - COMPREHENSIVE OVERVIEW")
print("=" * 60)

# Use uniformed + civilian (hourly only, OT meaningful)
df_ot = df_hourly_final.copy()

# Core OT metrics
total_payroll    = df_ot['Total_Compensation'].sum()
total_ot         = df_ot['Total OT Paid'].sum()
workers_with_ot  = (df_ot['Total OT Paid'] > 0).sum()
workers_no_ot    = (df_ot['Total OT Paid'] == 0).sum()

print(f"\nTotal Payroll:          ${total_payroll:>15,.0f}")
print(f"Total OT Paid:          ${total_ot:>15,.0f}")
print(f"OT % of Payroll:        {total_ot/total_payroll*100:>14.1f}%")
print(f"\nWorkers WITH OT:        {workers_with_ot:>15,} ({workers_with_ot/len(df_ot)*100:.1f}%)")
print(f"Workers WITHOUT OT:     {workers_no_ot:>15,} ({workers_no_ot/len(df_ot)*100:.1f}%)")
print(f"\nAvg OT (all workers):   ${df_ot['Total OT Paid'].mean():>15,.0f}")
print(f"Avg OT (OT workers):    ${df_ot[df_ot['Total OT Paid'] > 0]['Total OT Paid'].mean():>15,.0f}")
print(f"Median OT (OT workers): ${df_ot[df_ot['Total OT Paid'] > 0]['Total OT Paid'].median():>15,.0f}")
print(f"Max OT Paid:            ${df_ot['Total OT Paid'].max():>15,.0f}")

OVERTIME ANALYSIS - COMPREHENSIVE OVERVIEW

Total Payroll:          $145,546,220,436
Total OT Paid:          $ 17,803,306,148
OT % of Payroll:                  12.2%

Workers WITH OT:              1,143,716 (72.1%)
Workers WITHOUT OT:             442,884 (27.9%)

Avg OT (all workers):   $         11,221
Avg OT (OT workers):    $         15,566
Median OT (OT workers): $         10,295
Max OT Paid:            $        527,532


In [24]:
print("\n" + "=" * 60)
print("OT INTENSITY BY SEGMENT")
print("=" * 60)

for name, segment in [('Uniformed', df_uniformed), ('Civilian', df_civilian)]:
    seg_ot = segment[segment['Total OT Paid'] > 0]
    print(f"\n{name}:")
    print(f"  Workers with OT:      {len(seg_ot):,} ({len(seg_ot)/len(segment)*100:.1f}%)")
    print(f"  Avg OT (OT workers):  ${seg_ot['Total OT Paid'].mean():,.0f}")
    print(f"  Median OT:            ${seg_ot['Total OT Paid'].median():,.0f}")
    print(f"  OT % of Comp:         {segment['Total OT Paid'].sum()/segment['Total_Compensation'].sum()*100:.1f}%")
    print(f"  Max OT:               ${segment['Total OT Paid'].max():,.0f}")


OT INTENSITY BY SEGMENT

Uniformed:
  Workers with OT:      750,299 (93.5%)
  Avg OT (OT workers):  $19,623
  Median OT:            $15,159
  OT % of Comp:         17.2%
  Max OT:               $268,170

Civilian:
  Workers with OT:      345,474 (50.8%)
  Avg OT (OT workers):  $8,511
  Median OT:            $4,173
  OT % of Comp:         5.7%
  Max OT:               $527,532


In [25]:
print("\n" + "=" * 60)
print("TOP 15 AGENCIES BY TOTAL OT SPEND")
print("=" * 60)

ot_by_agency = (df_ot.groupby('Agency_Std')
    .agg(
        Total_OT       = ('Total OT Paid', 'sum'),
        Avg_OT         = ('Total OT Paid', 'mean'),
        Workers        = ('Total OT Paid', 'count'),
        OT_Workers     = ('Total OT Paid', lambda x: (x > 0).sum()),
        Total_Comp     = ('Total_Compensation', 'sum')
    )
    .assign(
        OT_Pct_Payroll = lambda x: x['Total_OT'] / x['Total_Comp'] * 100,
        OT_Worker_Pct  = lambda x: x['OT_Workers'] / x['Workers'] * 100
    )
    .sort_values('Total_OT', ascending=False)
    .head(15)
)

print(f"\n{'Agency':<35} {'Total OT':>12} {'OT% Payroll':>12} {'% Workers w/OT':>15}")
print("-" * 78)
for agency, row in ot_by_agency.iterrows():
    print(f"{agency:<35} ${row['Total_OT']:>11,.0f} {row['OT_Pct_Payroll']:>11.1f}% {row['OT_Worker_Pct']:>14.1f}%")


TOP 15 AGENCIES BY TOTAL OT SPEND

Agency                                  Total OT  OT% Payroll  % Workers w/OT
------------------------------------------------------------------------------
Police Department                   $7,417,528,572        15.2%           93.3%
Fire Department                     $3,722,703,361        20.5%           94.2%
Department of Correction            $2,019,958,632        21.0%           92.5%
Department of Sanitation            $1,562,956,902        17.4%           94.5%
Hra/Dept Of Social Services         $526,457,667         7.0%           53.8%
Nyc Housing Authority               $466,793,108         9.4%           69.3%
Administration for Children's Services $407,011,933         8.8%           64.3%
Department of Transportation        $378,352,951        12.6%           74.9%
Dept Of Environment Protection      $175,065,750         7.1%           61.4%
Department of Homeless Services     $145,182,150        11.8%           81.4%
Department of Ed

In [26]:
print("\n" + "=" * 60)
print("OT TREND 2015-2024")
print("=" * 60)

ot_trend = (df_ot.groupby('Fiscal Year')
    .agg(
        Total_OT   = ('Total OT Paid', 'sum'),
        Avg_OT     = ('Total OT Paid', 'mean'),
        Total_Comp = ('Total_Compensation', 'sum'),
        Headcount  = ('Total OT Paid', 'count')
    )
    .assign(OT_Pct = lambda x: x['Total_OT'] / x['Total_Comp'] * 100)
)

print(f"\n{'Year':>6} {'Total OT':>14} {'Avg OT':>10} {'OT % Payroll':>13} {'Headcount':>10}")
print("-" * 58)
for year, row in ot_trend.iterrows():
    print(f"{year:>6} ${row['Total_OT']:>13,.0f} ${row['Avg_OT']:>9,.0f} {row['OT_Pct']:>12.1f}% {row['Headcount']:>10,}")


OT TREND 2015-2024

  Year       Total OT     Avg OT  OT % Payroll  Headcount
----------------------------------------------------------
  2015 $1,473,205,619 $    9,634         11.8%  152,918.0
  2016 $1,588,150,562 $   10,170         12.2%  156,160.0
  2017 $1,666,630,350 $   10,302         11.8%  161,774.0
  2018 $1,617,304,106 $    9,770         11.5%  165,537.0
  2019 $1,567,109,998 $    9,485         11.0%  165,227.0
  2020 $1,653,482,218 $    9,754         11.0%  169,512.0
  2021 $1,492,422,592 $    9,561         10.7%  156,094.0
  2022 $2,050,794,172 $   13,405         13.5%  152,985.0
  2023 $2,131,100,694 $   14,061         13.9%  151,565.0
  2024 $2,563,105,839 $   16,555         14.1%  154,828.0


In [27]:
print("\n" + "=" * 60)
print("OT HIGH EARNERS ANALYSIS")
print("=" * 60)

# OT brackets
df_ot['OT_Bracket'] = pd.cut(
    df_ot['Total OT Paid'],
    bins=[0, 1, 10000, 25000, 50000, 100000, float('inf')],
    labels=['No OT', 'Low (<$10k)', 'Medium ($10-25k)', 
            'High ($25-50k)', 'Very High ($50-100k)', 'Extreme (>$100k)']
)

bracket_summary = (df_ot.groupby('OT_Bracket', observed=True)
    .agg(Workers=('Total OT Paid', 'count'),
         Avg_OT =('Total OT Paid', 'mean'),
         Total_OT=('Total OT Paid', 'sum'))
    .assign(Worker_Pct=lambda x: x['Workers']/len(df_ot)*100)
)

print(f"\n{'OT Bracket':<22} {'Workers':>10} {'% Workers':>10} {'Avg OT':>12} {'Total OT':>14}")
print("-" * 72)
for bracket, row in bracket_summary.iterrows():
    print(f"{str(bracket):<22} {row['Workers']:>10,} {row['Worker_Pct']:>9.1f}% "
          f"${row['Avg_OT']:>11,.0f} ${row['Total_OT']:>13,.0f}")


OT HIGH EARNERS ANALYSIS

OT Bracket                Workers  % Workers       Avg OT       Total OT
------------------------------------------------------------------------
No OT                     3,467.0       0.2% $          0 $        1,649
Low (<$10k)             558,942.0      35.2% $      3,684 $2,059,120,152
Medium ($10-25k)        326,482.0      20.6% $     16,511 $5,390,678,333
High ($25-50k)          206,319.0      13.0% $     34,552 $7,128,755,339
Very High ($50-100k)     46,019.0       2.9% $     63,568 $2,925,340,050
Extreme (>$100k)          2,487.0       0.2% $    120,390 $  299,410,625


OT is not exceptional for uniformed staff — it's built into their compensation model

In [28]:
# Who earned $527k in OT? Must investigate
print(df_civilian.nlargest(5, 'Total OT Paid')[
    ['Agency_Std', 'Title_Std', 'Fiscal Year',
     'Base Salary', 'Total OT Paid', 'Total_Compensation']
])


                           Agency_Std              Title_Std  Fiscal Year  \
2748273  Department of Transportation  Chief Marine Engineer         2024   
2748285  Department of Transportation  Chief Marine Engineer         2024   
2748276  Department of Transportation  Chief Marine Engineer         2024   
2748274  Department of Transportation  Chief Marine Engineer         2024   
2748269  Department of Transportation  Chief Marine Engineer         2024   

         Base Salary  Total OT Paid  Total_Compensation  
2748273     169520.0      527532.00          1411471.88  
2748285     169520.0      498669.50          1689517.92  
2748276     169520.0      479668.38          1437385.60  
2748274     169520.0      450520.20          1356513.20  
2748269     169520.0      437979.53          1191687.00  


In [29]:
# Check BOE by year - should spike in election years
boe_trend = (df_hourly_final[df_hourly_final['Agency_Std'] == 'Board Of Election']
    .groupby('Fiscal Year')['Total OT Paid']
    .agg(['sum', 'mean', 'count']))
print(boe_trend)

                     sum          mean  count
Fiscal Year                                  
2015          3725263.59  10319.289723    361
2016          5110452.70  11207.133114    456
2017          6777262.27  13608.960382    498
2018          5529164.82  10612.600422    521
2019          9424785.18  17782.613547    530
2020          8886455.05  13423.648112    662
2021         16127841.80  22811.657426    707
2022         10850413.06  15702.479103    691
2023         10255132.83  15215.330608    674
2024         11676847.49  17046.492686    685


In [30]:
# How many Chief Marine Engineers exist and what's the pattern?
cme = df_civilian[df_civilian['Title_Std'] == 'Chief Marine Engineer']
print(f"Total Chief Marine Engineer records: {len(cme):,}")
print(f"\nBy Year:")
print(cme.groupby('Fiscal Year').agg(
    Count     = ('Total OT Paid', 'count'),
    Avg_OT    = ('Total OT Paid', 'mean'),
    Total_OT  = ('Total OT Paid', 'sum'),
    Avg_Comp  = ('Total_Compensation', 'mean')
).round(0))

Total Chief Marine Engineer records: 305

By Year:
             Count    Avg_OT   Total_OT  Avg_Comp
Fiscal Year                                      
2015            31   59100.0  1832109.0  132063.0
2016            34   59043.0  2007464.0  134791.0
2017            34   60992.0  2073726.0  138548.0
2018            31   58034.0  1799059.0  132974.0
2019            32   56878.0  1820088.0  132078.0
2020            29   51585.0  1495956.0  128092.0
2021            27   48714.0  1315286.0  132000.0
2022            29   54641.0  1584591.0  129852.0
2023            27   53546.0  1445746.0  130989.0
2024            31  243905.0  7561058.0  829482.0


In [31]:
# Combine all findings into era summary
print("=" * 65)
print("EXTREME OT EARNERS (>$100k OT) - COVID IMPACT")
print("=" * 65)

extreme = df_hourly_final[df_hourly_final['Total OT Paid'] > 100000].copy()

# By agency and year
extreme_agency_year = (extreme.groupby(['Fiscal Year', 'Agency_Std'])
    .agg(Workers=('Total OT Paid', 'count'),
         Avg_OT =('Total OT Paid', 'mean'),
         Total_OT=('Total OT Paid', 'sum'))
    .sort_values(['Fiscal Year', 'Total_OT'], ascending=[True, False])
)

# Show top agency per year
print(f"\n{'Year':>6} {'Top Agency':<35} {'Workers':>8} {'Avg OT':>10} {'Total OT':>14}")
print("-" * 78)
for year in range(2015, 2025):
    year_data = extreme_agency_year.loc[year] if year in extreme_agency_year.index.get_level_values(0) else None
    if year_data is not None and len(year_data) > 0:
        top = year_data.iloc[0]
        agency = year_data.index[0]
        print(f"{year:>6} {agency:<35} {top['Workers']:>8,.0f} "
              f"${top['Avg_OT']:>9,.0f} ${top['Total_OT']:>13,.0f}")

EXTREME OT EARNERS (>$100k OT) - COVID IMPACT

  Year Top Agency                           Workers     Avg OT       Total OT
------------------------------------------------------------------------------
  2015 Fire Department                           12 $  117,450 $    1,409,401
  2016 Department of Correction                  55 $  115,321 $    6,342,630
  2017 Department of Correction                  59 $  116,265 $    6,859,661
  2018 Department of Correction                  33 $  113,633 $    3,749,904
  2019 Fire Department                            8 $  117,577 $      940,620
  2020 Department of Correction                   8 $  116,553 $      932,426
  2021 Department of Sanitation                  94 $  116,395 $   10,941,163
  2022 Department of Correction                 136 $  122,488 $   16,658,315
  2023 Department of Correction                 274 $  124,044 $   33,988,084
  2024 Fire Department                          505 $  117,665 $   59,420,589


In [32]:
print("\n" + "="*65)
print("WHAT DROVE THE 2021→2022 OT SURGE (+37.4%)?")
print("="*65)

ot_2021 = (df_hourly_final[df_hourly_final['Fiscal Year']==2021]
           .groupby('Agency_Std')['Total OT Paid'].sum())
ot_2022 = (df_hourly_final[df_hourly_final['Fiscal Year']==2022]
           .groupby('Agency_Std')['Total OT Paid'].sum())

ot_change = (pd.DataFrame({'OT_2021': ot_2021, 'OT_2022': ot_2022})
             .fillna(0)
             .assign(
                 Change     = lambda x: x['OT_2022'] - x['OT_2021'],
                 Change_Pct = lambda x: (x['OT_2022'] / x['OT_2021'].replace(0, float('nan')) - 1) * 100
             )
             .sort_values('Change', ascending=False))

print(f"\n{'Agency':<35} {'2021 OT':>13} {'2022 OT':>13} {'Change':>12} {'Chg%':>7}")
print("-" * 83)
for agency, row in ot_change.head(12).iterrows():
    print(f"{agency:<35} ${row['OT_2021']:>12,.0f} ${row['OT_2022']:>12,.0f} "
          f"${row['Change']:>11,.0f} {row['Change_Pct']:>+6.1f}%")

# Also check headcount change same period
print(f"\n{'HEADCOUNT CHANGE 2021→2022':^65}")
hc_2021 = df_hourly_final[df_hourly_final['Fiscal Year']==2021]['Agency_Std'].value_counts()
hc_2022 = df_hourly_final[df_hourly_final['Fiscal Year']==2022]['Agency_Std'].value_counts()
hc_change = pd.DataFrame({'HC_2021': hc_2021, 'HC_2022': hc_2022}).fillna(0)
hc_change['Change'] = hc_change['HC_2022'] - hc_change['HC_2021']
print(hc_change.sort_values('Change').head(10).to_string())


WHAT DROVE THE 2021→2022 OT SURGE (+37.4%)?

Agency                                    2021 OT       2022 OT       Change    Chg%
-----------------------------------------------------------------------------------
Police Department                   $ 465,940,004 $ 742,748,074 $276,808,070  +59.4%
Fire Department                     $ 334,827,964 $ 473,825,592 $138,997,628  +41.5%
Department of Correction            $ 127,950,757 $ 229,556,403 $101,605,646  +79.4%
Hra/Dept Of Social Services         $  51,045,763 $  78,439,662 $ 27,393,899  +53.7%
Administration for Children's Services $  22,340,775 $  35,520,747 $ 13,179,972  +59.0%
Department of Education             $   9,267,374 $  16,092,375 $  6,825,001  +73.6%
Nyc Housing Authority               $  50,216,445 $  55,670,795 $  5,454,350  +10.9%
Department of Transportation        $  32,762,243 $  37,418,880 $  4,656,637  +14.2%
Dept Of Environment Protection      $  13,541,638 $  17,462,898 $  3,921,260  +29.0%
Department of Par

Correction:  -1,430 workers  → +79.4% OT increase
Police:      -1,178 workers  → +59.4% OT increase
Every major agency LOST headcount but GAINED OT

In [33]:
# Correction deep dive
correction = df_uniformed[df_uniformed['Agency_Std'] == 'Department of Correction']
print(correction.groupby('Fiscal Year').agg(
    Headcount   = ('Total OT Paid', 'count'),
    Total_OT    = ('Total OT Paid', 'sum'),
    Avg_OT      = ('Total OT Paid', 'mean'),
    Extreme_OT  = ('Total OT Paid', lambda x: (x > 100000).sum()),
    OT_Pct_Comp = ('Total OT Paid', lambda x: x.sum() / 
                   correction.loc[x.index, 'Total_Compensation'].sum() * 100)
).round(0))


             Headcount     Total_OT   Avg_OT  Extreme_OT  OT_Pct_Comp
Fiscal Year                                                          
2015              8927  168874397.0  18917.0           6         19.0
2016              9113  255097855.0  27993.0          55         26.0
2017             10350  243400517.0  23517.0          59         24.0
2018             10595  195715410.0  18472.0          33         20.0
2019             10630  155695117.0  14647.0           5         16.0
2020             10335  128536750.0  12437.0           8         13.0
2021              9216  127950757.0  13884.0          13         14.0
2022              7786  229556403.0  29483.0         136         24.0
2023              7034  264554127.0  37611.0         274         28.0
2024              6740  250577298.0  37178.0         291         27.0


In [34]:
# Fire Department trajectory
fire = df_uniformed[df_uniformed['Agency_Std'] == 'Fire Department']
print(fire.groupby('Fiscal Year').agg(
    Headcount  = ('Total OT Paid', 'count'),
    Total_OT   = ('Total OT Paid', 'sum'),
    Avg_OT     = ('Total OT Paid', 'mean'),
    Extreme_OT = ('Total OT Paid', lambda x: (x > 100000).sum())
).round(0))


             Headcount     Total_OT   Avg_OT  Extreme_OT
Fiscal Year                                             
2015             14533  305745221.0  21038.0          12
2016             15067  332464122.0  22066.0           1
2017             15532  309899206.0  19952.0           2
2018             16329  315049984.0  19294.0           6
2019             15166  305138552.0  20120.0           8
2020             16569  315593977.0  19047.0           3
2021             16116  334827964.0  20776.0          10
2022             15880  473825592.0  29838.0          53
2023             15825  470978596.0  29762.0         144
2024             16109  559180145.0  34712.0         505


NYC Open DATA Site info NOTE 1: To further improve the visibility into the number of employee OT hours worked, beginning with the FY 2023 report, an updated methodology will be used which will eliminate redundant reporting of OT hours in some specific instances. In the previous calculation, hours associated with both overtime pay as well as an accompanying overtime “companion code” pay were included in the employee total even though they represented pay for the same period of time. With the updated methodology, the dollars shown on the Open Data site will continue to be inclusive of both types of overtime, but the OT hours will now reflect a singular block of time, which will result in a more representative total of employee OT hours worked. The updated methodology will primarily impact the OT hours associated with City employees in uniformed civil service titles. The updated methodology will be applied to the Open Data posting for Fiscal Year 2023 and cannot be applied to prior postings and, as a result, the reader of this data should not compare OT hours prior to the 2023 report against OT hours published starting Fiscal Year 2023. The reader of this data may continue to compare OT dollars across all published Fiscal Years on Open Data. this data description mean and impact?

# OT HOURS cannot be compared pre vs post 2023
df['OT Hours']   # ❌ 2015-2022 = inflated (double counted)
                 # ❌ 2023-2024 = corrected (true hours)
                 # ❌ NEVER compare these directly

In [35]:
# Add a flag to prevent accidental OT hours comparison
df['OT_Hours_Reliable'] = df['Fiscal Year'] >= 2023

# If you ever use OT hours, always filter or caveat
def safe_ot_hours_analysis(df, metric='OT Hours'):
    pre_2023  = df[df['Fiscal Year'] < 2023]
    post_2023 = df[df['Fiscal Year'] >= 2023]
    
    print("⚠️  OT HOURS METHODOLOGY BREAK AT FY2023")
    print("    Pre-2023 hours are INFLATED (double counted)")
    print("    Post-2023 hours are CORRECTED")
    print(f"\n    Pre-2023  avg OT hours: {pre_2023[metric].mean():.1f}  ← overstated")
    print(f"    Post-2023 avg OT hours: {post_2023[metric].mean():.1f}  ← reliable")
    print("\n    Use OT DOLLARS for cross-year comparisons instead")

# Also add to your documentation
print("ANALYSIS NOTE: All OT comparisons use OT DOLLARS not OT HOURS")
print("Reason: FY2023 methodology change makes hours non-comparable")

ANALYSIS NOTE: All OT comparisons use OT DOLLARS not OT HOURS
Reason: FY2023 methodology change makes hours non-comparable


In [36]:
# Ensure output directory exists
from pathlib import Path
output_path = repo_root / 'data' / 'processed'
output_path.mkdir(parents=True, exist_ok=True)

# Files to save (your DataFrames)
files = {
    'nyc_payroll_hourly_employees_2015_2024.csv':   df_hourly_final,
    'nyc_payroll_salaried_doe_2015_2024.csv':      df_salaried_final,
    'nyc_payroll_combined_all_2015_2024.csv':     df,
}

# Save each CSV (use compression to save space)
for name, df_out in files.items():
    path = output_path / name
    # If DataFrame is reasonably sized:
    df_out.to_csv(path, index=False)
    # If you prefer compressed files:
    # df_out.to_csv(str(path) + '.gz', index=False, compression='gzip')
    print(f"Saved {path}  (rows: {len(df_out):,}, cols: {len(df_out.columns):,})")

Saved /Users/mehreen.gillaniicloud.com/Downloads/NYC_Payroll/data622/data/processed/nyc_payroll_hourly_employees_2015_2024.csv  (rows: 1,586,600, cols: 21)
Saved /Users/mehreen.gillaniicloud.com/Downloads/NYC_Payroll/data622/data/processed/nyc_payroll_salaried_doe_2015_2024.csv  (rows: 1,031,042, cols: 21)
Saved /Users/mehreen.gillaniicloud.com/Downloads/NYC_Payroll/data622/data/processed/nyc_payroll_combined_all_2015_2024.csv  (rows: 2,617,642, cols: 22)
